# 16.12 滑動窗口省下什麼，也限制了什麼？


你讀到一封長信的最後一句，想查前面提過的日期。完整的因果注意力允許這個位置比對所有前文。若信很長，這次比對和保留前文的快取也會變大。一種取捨是：每個位置只直接查看附近幾格，讓需要比對的範圍有固定上限。

用八個位置看這件事。位置從 0 編到 7；這些數字表示前後次序，並不是字的 ID。我們規定一個窗口包含「自己，加上前面最多兩格」，所以窗口寬度為 3。位置 7 在這一層能讀 5、6、7，不能直接讀 4。位置 5 則能讀 3、4、5。這種隨著目前位置一起向前移動的可見範圍，叫作**滑動窗口注意力**。

窗口寬度要先說清楚。有的文件把寬度算成過去幾格，有的包含目前格；本節採後者。知道這個約定，才不會把同一張圖算差一格。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/rewrite-16-12-sliding-path.svg")))

圖中最下面的位置 7 是第 2 層的特徵。它直接讀第 1 層的 5、6、7，也就是三個橘框。第 1 層的位置 5 已經讀過輸入的 3、4、5，因此輸入位置 3 的資訊有機會沿著「3 → 5 → 7」傳過來。兩層合起來，最後位置可能受到 3 至 7 的影響，雖然每層直接讀的仍然只有三格。

這像把一段話交給旁邊的人，再由他轉述給你。訊息可能傳到更遠，但你收到的是中間特徵帶來的資訊，不能保證保留每個字，也不能把這條傳遞路徑當成模型一定記得日期的證據。窗口外的原文若已從快取移除，後來的位置也不能直接回頭比對它。

我們可以只數可見的配對，先不測模型速度：


In [ ]:
length = 8
window = 3  # 包含目前位置


def visible(position):
    return set(range(max(0, position - window + 1), position + 1))


print("位置7直接讀", sorted(visible(7)))
two_layers = set().union(*(visible(p) for p in visible(7)))
print("兩層可能傳來的來源", sorted(two_layers))
print("完整因果配對", sum(p + 1 for p in range(length)))
print("滑窗配對", sum(len(visible(p)) for p in range(length)))

`visible`回傳一個集合，記錄可讀位置而不重複計數。計算`two_layers`時，`visible(7)`先給出5、6、7；`for p`逐一取這三個位置，再用`visible(p)`得到三個來源集合：3至5、4至6、5至7。外面的`*`把這三個集合展開成`union`的三個參數；`set()`是空集合，`union`把來源合併並去掉重複位置。因此前兩行分別是 `[5, 6, 7]` 與 `[3, 4, 5, 6, 7]`。八個位置的完整因果配對有 36 組，滑窗有 21 組：第一格只有自己，第二格有兩格，其餘六格各有三格。這是規則的計數，沒有證明任何模型答對，也不是實際耗時的倍數。

當序列很長、窗口寬度固定，單層注意力的配對數便大致隨「長度 × 窗口寬度」增加，完整因果注意力則隨長度的平方增加。若實作也只保留窗口內的 K、V，生成時每層的這部分快取能保持有界。省下的範圍來自運算和儲存方式；只在完整大矩陣上塗掉格子，未必真的省到同樣成本。

因此，滑窗的選擇包含兩件事：減少直接比對的範圍，以及接受較早資訊需要經過其他位置傳遞的限制。需要回查久遠細節的任務，要另外驗收，或考慮檢索、部分全域層等設計。延長可接受的輸入長度，和穩定使用那段輸入，是不同的要求。

試著把 `window` 改成 2。位置 7 直接讀哪幾格？兩層最早可能受到哪一格影響？先用圖的走法判斷，再用程式核對；答案應為直接讀 6、7，兩層可能受到 5、6、7 的影響。

<details>
<summary>來源與延伸</summary>

[Mistral 7B 原論文第 2 節](https://arxiv.org/pdf/2310.06825v1)分別說明滑動窗口、滾動快取與分塊讀取。本節用包含目前位置的寬度 3 示範連結，並未重現該模型的設定或成績。實際採用滑窗時，需核對窗口定義、位置編號、快取淘汰方式與所用核心是否真的減少工作。

</details>
